LSTM TEXT CLASSIFICATION

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [20]:
# Load dataset
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

In [21]:
# Padding
max_len = 200
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [22]:
# Model
model = Sequential()
model.add(Embedding(vocab_size, 64, input_length=max_len))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

In [23]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
# Train
model.fit(X_train, y_train, epochs=3, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 81s 201ms/step - accuracy: 0.7937 - loss: 0.4285 - val_accuracy: 0.8218 - val_loss: 0.4192
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 84s 207ms/step - accuracy: 0.8966 - loss: 0.2623 - val_accuracy: 0.8670 - val_loss: 0.3175
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 80s 203ms/step - accuracy: 0.9294 - loss: 0.1835 - val_accuracy: 0.8644 - val_loss: 0.3477


SEQ2SEQ MODEL

In [25]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

In [26]:
# Load IMDB
vocab_size = 10000
(X_train, y_train), _ = imdb.load_data(num_words=vocab_size)

In [27]:
# Padding
max_len = 100
X_train = pad_sequences(X_train, maxlen=max_len)

In [28]:
# Convert labels to sequence
y_seq = np.repeat(y_train.reshape(-1,1), max_len, axis=1)
y_seq = np.expand_dims(y_seq, -1)

In [30]:
from tensorflow.keras.layers import Embedding

#encoding
encoder_inputs = Input(shape=(max_len,))
encoder_embed = Embedding(10000, 64)(encoder_inputs)

encoder = LSTM(64, return_state=True)
_, state_h, state_c = encoder(encoder_embed)

In [31]:
# Decoder
decoder_inputs = Input(shape=(max_len,1))
decoder_lstm = LSTM(64, return_sequences=True)
decoder_outputs = decoder_lstm(decoder_inputs, initial_state=[state_h, state_c])

In [32]:
decoder_dense = Dense(1, activation='sigmoid')
decoder_outputs = decoder_dense(decoder_outputs)

In [33]:
# Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='binary_crossentropy')

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 100, 64)   │    640,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, 64),      │     33,024 │ embedding_2[0][0] │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 100, 64)   │     16,896 │ input_layer_7[0]… │
│                     │                   │            │ lstm_6[0][1],     │
│                     │                   │            │ lstm_6[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 100, 1)    │         65 │ lstm_7[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 689,985 (2.63 MB)

 Trainable params: 689,985 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
# Train
model.fit([X_train, y_seq], y_seq, epochs=2, batch_size=64)

Epoch 1/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 59s 139ms/step - loss: 0.0613
Epoch 2/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 82s 138ms/step - loss: 0.0061


In [35]:
sample_input = X_train[:1]
sample_target = y_seq[:1]

pred = model.predict([sample_input, sample_target])

print("Actual label sequence:", sample_target[0][:10])
print("Predicted sequence:", pred[0][:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 375ms/step
Actual label sequence: [[1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]]
Predicted sequence: [[0.7076521 ]
 [0.93300194]
 [0.9950467 ]
 [0.99930924]
 [0.99966186]
 [0.9997508 ]
 [0.99978137]
 [0.9997932 ]
 [0.9997982 ]
 [0.9998004 ]]
